# Ionic Dielectric Constant with VASP

The ionic or lattice response to an external electric field is one of the two contributions to the total "static" dielectric constant. The electronic contribution $\varepsilon_\infty$ is obtained from the optics calculation, but the ionic contribution $\varepsilon_\mathrm{ion}$ requires a calculation of its own. The total is their sum:

$$\varepsilon_\mathrm{static} = \varepsilon_\infty + \varepsilon_\mathrm{ion}$$

`solphin` provides methods to write input files and read output files to calculate this using VASP.

There are two possible methods:

1) **Density Functional Perturbation Theory (DFPT)** - much cheaper and faster. VASP does not implement DFPT for meta-GGA or hybrid functionals, so this route needs a GGA such as PBE, which is typically an acceptable approximation. (Recommended)

2) **The finite-response approach** - computes the response to a finite electric field instead of a perturbation, so it supports any computational regime including hybrids, but is slower and more expensive.

Pre-requisites:
* Geometry Optimisation/Structural Relaxation

## Method 1 - DFPT

1) Import the necessary libraries.

In [ ]:
import solphin.dielectric as dielectric
import solphin.vasp_inputs as vasp_inputs

2) Define our computational regime. The encut and k-spacing should match our converged values, but we are going to use a GGA functional like PBE for this calculation, unlike the hybrid functional we used for the rest of the tutorials.

In [ ]:
functional = 'PBE'    # The DFT functional
encut = 450           # Plane-wave basis set energy cutoff
kspacing = 0.2        # k-point sampling density

3) We need to make sure the structure is at an equilibrium under our new regime, so we need to run a new relaxation. By starting from our previous relaxed structure, this shouldn't take as long.

* If you have been using a GGA functional already, you can skip this step.

* Here we reference the relaxed structure in the test data, and write to the untracked `workdir`.

In [ ]:
structure = vasp_inputs.read_structure_pmg("../tests/data/Cu2GeS3/Relax/CONTCAR")

In [ ]:
vasp_inputs.write_vasp_calculation(
    structure=structure,
    recipe=functional,
    out_dir="workdir/ionic_dielectric_relax",
    patches=["relax_cell", "tight_relax"],
    user_incar_settings={"ENCUT": encut, "KSPACING": kspacing}
)

4) Run VASP in `workdir/ionic_dielectric_relax`.

* The relaxation matters more here than usual: the ionic dielectric constant comes from the phonon modes, and residual forces on the atoms produce imaginary modes and a meaningless tensor. That is why we use the `tight_relax` patch.

5) With a structure relaxed under GGA, we can set up the DFPT calculation. Note that we read the `CONTCAR`, the *result* of the relaxation, rather than the `POSCAR` we wrote as its input.

In [ ]:
gga_structure = vasp_inputs.read_structure_pmg("workdir/ionic_dielectric_relax/CONTCAR")

In [ ]:
vasp_inputs.write_vasp_calculation(
    structure=gga_structure,
    recipe=functional,
    out_dir="workdir/ionic_dielectric_dfpt",
    patches=["dfpt"],
    user_incar_settings={"ENCUT": encut, "KSPACING": kspacing}
)

6) Run VASP in `workdir/ionic_dielectric_dfpt`.

7) Once the calculation is complete, we can parse the ionic dielectric constant. `parse_ionic_dielectric` returns both the diagonal average and the full 3×3 tensor:

In [ ]:
eps_ion, eps_ion_tensor = dielectric.parse_ionic_dielectric("workdir/ionic_dielectric_dfpt/OUTCAR")

print(f"Ionic contribution: {eps_ion:.3f}")
print(eps_ion_tensor)

**IMPORTANT** You should take care to ensure the convergence of the dielectric constant with respect to the k-point sampling (k-spacing). This often requires a denser mesh than a regular calculation.

## Method 2 - Finite Response

This route computes the response to a finite electric field (`LCALCEPS`) rather than to a perturbation, so unlike DFPT it works with hybrid functionals, meta-GGAs and Van-der-Waals corrections. It is correspondingly more expensive.

Use this method if you need the dielectric constant in the same regime as the rest of your calculations.

1) Import the necessary libraries.

In [ ]:
import solphin.dielectric as dielectric
import solphin.vasp_inputs as vasp_inputs

2) Define our computational regime. This method works with any regime, so use the same one you have been using.

In [ ]:
functional = 'HSE06'  # The DFT functional
encut = 450           # Plane-wave basis set energy cutoff
kspacing = 0.2        # k-point sampling density

3) Load a structure relaxed in **this** regime. The same caution about residual forces applies, so this should be the `CONTCAR` of a tight relaxation run with the same functional.

In [ ]:
structure = vasp_inputs.read_structure_pmg("../tests/data/Cu2GeS3/Relax/CONTCAR")

4) Write the inputs for the calculation:

In [ ]:
vasp_inputs.write_vasp_calculation(
    structure=structure,
    recipe=functional,
    out_dir="workdir/ionic_dielectric_finite",
    patches=["lattice_response"],
    user_incar_settings={"ENCUT": encut, "KSPACING": kspacing}
)

5) Run VASP in `workdir/ionic_dielectric_finite`.

6) Once the calculation is complete, we can parse the ionic dielectric constant.

In [ ]:
eps_ion, eps_ion_tensor = dielectric.parse_ionic_dielectric(
    "workdir/ionic_dielectric_finite/OUTCAR"
)

print(f"Ionic contribution: {eps_ion:.3f}")
print(eps_ion_tensor)

**IMPORTANT** You should take care to ensure the convergence of the dielectric constant with respect to the k-point sampling (k-spacing). This often requires a denser mesh than a regular calculation.